In [1]:
import os
os.environ["FLAGS_enable_pir_api"] = "0"
os.environ["FLAGS_use_mkldnn"] = "0"   # also disable oneDNN/mkldnn, the actual culprit

# now import and init
from paddleocr import PaddleOCR
ocr = PaddleOCR(use_textline_orientation=True, lang='ar')  # note: new arg name (3.x)
print("initialized")

C:\Users\aksha\anaconda3\envs\paddleocr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\aksha\anaconda3\envs\paddleocr\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\aksha\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\aksha\.paddlex\official_model

initialized


In [2]:
# Test on ONE KITAB-Bench image first, and inspect the return structure.
from datasets import load_dataset
import numpy as np

ds = load_dataset("ahmedheakl/arocrbench_synthesizear", split="train")
sample = ds[0]
img = sample["image"]            # PIL image
gt = sample["text"]              # ground truth

# PaddleOCR wants a numpy array (or a file path), not a PIL image
img_np = np.array(img)

# PaddleOCR 3.x: use .predict() ; 2.x used .ocr(). Try predict first.
result = ocr.predict(img_np)

print("=== Ground truth ===")
print(gt)
print("\n=== Raw result type ===")
print(type(result))
print("\n=== Raw result (inspect structure) ===")
print(result)

=== Ground truth ===
وَإِذا ما سَأَلَتْنِي عَن مَعْنَى لَفَظَهُ "عرب" عِنْدَ

=== Raw result type ===
<class 'list'>

=== Raw result (inspect structure) ===
[{'input_path': None, 'page_index': None, 'doc_preprocessor_res': {'input_path': None, 'page_index': None, 'input_img': array([[[255, ..., 255],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], shape=(114, 639, 3), dtype=uint8), 'model_settings': {'use_doc_orientation_classify': True, 'use_doc_unwarping': True}, 'angle': 0, 'rot_img': array([[[255, ..., 255],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], shape=(114, 639, 3), dtype=uint8), 'output_img': array([[[255, ..., 255],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], shape=(114, 639, 3), dtype=uint8)}, 'dt_polys': [array([[  1,  25],
       ...,
       [  4, 110]], sh

In [3]:
import numpy as np
from datasets import load_dataset
import jiwer

ds = load_dataset("ahmedheakl/arocrbench_synthesizear", split="train")

def paddle_ocr_text(pil_img):
    """Run PaddleOCR, return joined text of all detected lines + min line score."""
    img_np = np.array(pil_img)
    result = ocr.predict(img_np)
    if not result or not result[0].get('rec_texts'):
        return "", 0.0
    rec_texts = result[0]['rec_texts']
    rec_scores = result[0]['rec_scores']
    text = " ".join(rec_texts)
    min_score = min(rec_scores) if rec_scores else 0.0
    return text, min_score

# Run PaddleOCR on the same 50 samples
N = 50
paddle_results = []
print(f"Running PaddleOCR on {N} samples...")
for i in range(N):
    sample = ds[i]
    gt = sample["text"]
    text, score = paddle_ocr_text(sample["image"])
    paddle_results.append({"index": i, "gt": gt, "paddle_text": text, "paddle_score": score})
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{N} done")

print("PaddleOCR run complete.")

Running PaddleOCR on 50 samples...
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done
PaddleOCR run complete.


NameError: name 'json' is not defined

In [4]:
import json
with open("paddle_50_outputs.json", "w", encoding="utf-8") as f:
    json.dump(paddle_results, f, ensure_ascii=False, indent=2)
print("Saved paddle_50_outputs.json")

Saved paddle_50_outputs.json


In [5]:
import json, jiwer

surya = {r["index"]: r for r in json.load(open("surya_50_outputs.json", encoding="utf-8"))}
paddle = {r["index"]: r for r in json.load(open("paddle_50_outputs.json", encoding="utf-8"))}

print(f"{'Idx':<5}{'SuryaCER':>9}{'Disagree':>10}{'S-conf':>8}  Note")
print("-"*70)
for i in range(50):
    s = surya[i]; p = paddle[i]
    disagreement = jiwer.cer(s["surya_text"], p["paddle_text"])  # engines vs each other
    note = ""
    if i == 43: note = "<- hallucinated repetition (perplexity blind)"
    if i == 13: note = "<- subtle error (evaded all 4 signals)"
    print(f"{i:<5}{s['surya_cer']*100:>8.1f}%{disagreement*100:>9.1f}%{s['surya_min_conf']:>8.2f}  {note}")

Idx   SuryaCER  Disagree  S-conf  Note
----------------------------------------------------------------------
0         1.8%     60.7%    0.97  
1         4.4%     59.4%    0.97  
2         0.0%     39.3%    1.00  
3         0.0%     68.8%    1.00  
4         0.0%     95.2%    1.00  
5         1.9%     60.4%    1.00  
6         8.8%     83.1%    0.92  
7         1.8%     58.9%    1.00  
8         0.0%     82.3%    0.99  
9         2.7%     76.0%    0.98  
10       21.1%     69.4%    0.46  
11        6.2%     82.5%    1.00  
12       75.0%    100.0%    0.48  
13       11.8%     78.9%    1.00  <- subtle error (evaded all 4 signals)
14       60.2%     80.6%    0.17  
15       28.6%     74.5%    0.91  
16        1.8%     28.6%    1.00  
17        4.2%     89.0%    1.00  
18        0.0%     73.9%    1.00  
19        4.3%     84.1%    0.63  
20        1.5%     47.0%    0.99  
21        0.0%     80.0%    0.97  
22        0.0%     75.9%    0.97  
23        4.1%     45.5%    0.99  
24       14.